# Muon Optimizer Development

This notebook is for developing and testing the Muon optimizer for use in model metamer generation.

In [48]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
from muon import MuonWithAuxAdam  # Make sure this import works!
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

## Test Muon optimizer
set up a simple test to ensure the optimizer is working as expected

In [3]:
# create an MLP model
model = torch.nn.Sequential(
    torch.nn.Linear(10, 10),
    torch.nn.ReLU(),
    torch.nn.Linear(10, 1)
)

In [ ]:
hidden_weights = [p for p in model.parameters() if p.ndim >= 2]
hidden_gains_biases = [p for p in model.parameters() if p.ndim < 2]

param_groups = [
    dict(params=hidden_weights, use_muon=True,
         lr=0.02, weight_decay=0.01),
    dict(params=hidden_gains_biases, use_muon=False,
         lr=3e-4, betas=(0.9, 0.95), weight_decay=0.01),
]

In [7]:
optimizer = MuonWithAuxAdam(param_groups)

In [ ]:
model_dir = 'model_analysis_folders/visual_networks/resnet50'

## Load pretrained model

In [42]:
# Example: Use ResNet18 and grab a late layer
model = models.resnet18(pretrained=True).eval()

# Choose a layer to match (e.g., the penultimate layer)
layer_name = 'avgpool'
activation = {}

def get_activation(name):
    def hook(model, input, output):
        # Don't detach during optimization - we need gradients!
        activation[name] = output
    return hook

# Register the hook
getattr(model, layer_name).register_forward_hook(get_activation(layer_name))

## Prepare a target image and get its activations

In [49]:
# Simple image loading function that mimics the metamer generation pipeline
# This avoids the numpy/numba compatibility issues

def load_image_like_metamer_script(image_path, data_format='NCHW'):
    """Load and preprocess image similar to the metamer generation script"""
    img_pil = Image.open(image_path)
    
    # Convert to RGB if needed
    if img_pil.mode != 'RGB':
        img_pil = img_pil.convert('RGB')
    
    # Center crop to square (like preproc_imagenet_center_crop)
    width, height = img_pil.size
    smallest_dim = min((width, height))
    left = (width - smallest_dim)/2
    right = (width + smallest_dim)/2
    top = (height - smallest_dim)/2
    bottom = (height + smallest_dim)/2
    img_pil = img_pil.crop((left, top, right, bottom))
    
    # Resize to 224x224
    img_pil = img_pil.resize((224, 224))
    img_pil.load()
    
    # Convert to numpy array
    img1 = np.asarray(img_pil, dtype="float32")
    
    # Convert to NCHW format if requested
    if data_format == 'NCHW':
        img1 = np.rollaxis(img1, 2, 0)
    
    # Ensure we have a proper numpy array with the right dtype
    img1 = np.asarray(img1, dtype=np.float32)
    
    # Create image dictionary similar to the metamer script
    image_dict = {
        'image': img1,
        'shape': 224,
        'filename': image_path,
        'filename_short': image_path.split('/')[-1],
        'max_value_image_set': 255,
        'min_value_image_set': 0
    }
    
    return image_dict

def preproc_image(image, image_dict):
    """The image into the pytorch model should be between 0-1"""
    if image_dict['max_value_image_set'] == 255:
        image = image / 255.
    return image

# Load the airplane image using our simple function
image_path = 'assets/full_400_16_class_imagenet_val_images/12_0_airplane_n02690373_00040136.JPEG'
image_dict = load_image_like_metamer_script(image_path, data_format='NCHW')
image_dict['image_orig'] = image_dict['image']

# Preprocess to be in the format for pytorch (0-1 range)
image_dict['image'] = preproc_image(image_dict['image'], image_dict)

# Debug information
print(f"Image dtype: {image_dict['image'].dtype}")
print(f"Image shape: {image_dict['image'].shape}")
print(f"Image range: [{image_dict['image'].min():.3f}, {image_dict['image'].max():.3f}]")

# Add a batch dimension to the input image 
# Use torch.FloatTensor to avoid numpy compatibility issues
image_array = np.expand_dims(image_dict['image'], 0)
input_img = torch.FloatTensor(image_array)

# Apply ImageNet normalization (this is crucial for pretrained models!)
# ImageNet normalization: mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
input_img = normalize(input_img)

print(f"Loaded image: {image_dict['filename_short']}")
print(f"Image shape: {input_img.shape}")
print(f"Image range after normalization: [{input_img.min():.3f}, {input_img.max():.3f}]")

# Note: Target activations will be computed after moving to GPU

Image dtype: float32
Image shape: (3, 224, 224)
Image range: [0.000, 1.000]
Loaded image: 12_0_airplane_n02690373_00040136.JPEG
Image shape: torch.Size([1, 3, 224, 224])
Image range after normalization: [-2.118, 2.640]


In [50]:
# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Initialize distributed environment for single-GPU Muon optimizer
import os
import torch.distributed as dist

# Set up environment variables for local single-GPU distributed training
os.environ['MASTER_ADDR'] = 'localhost'
os.environ['MASTER_PORT'] = '29500'

# Initialize process group for single GPU
try:
    dist.init_process_group(
        backend='nccl' if device.type == 'cuda' else 'gloo',
        init_method='env://',
        world_size=1,  # Single process
        rank=0
    )
    print("Distributed process group initialized for single GPU")
except Exception as e:
    print(f"Note: Could not initialize distributed group: {e}")
    print("Will try alternative approach...")

# Move model and inputs to GPU
model = model.to(device)
input_img = input_img.to(device)

# Get target activations on GPU
with torch.no_grad():
    model(input_img)
target_activations = activation[layer_name].clone()

print(f"Target activations device: {target_activations.device}")
print(f"Target activations shape: {target_activations.shape}")

# Start from random noise - properly initialize with gradient computation on GPU
# Store the original image shape for reshaping later
original_img_shape = input_img.shape
print(f"Original image shape: {original_img_shape}")

# Initialize synthetic image in the same range as normalized ImageNet images
# ImageNet normalized images typically range from about -2.1 to 2.6
# We'll start with random noise in a reasonable range around the target image
synth_img_4d = torch.randn_like(input_img, device=device) * 0.1 + input_img.mean()
synth_img_4d.requires_grad_(True)

# For Muon optimizer, try flattening the image to avoid shape issues
synth_img_flat = torch.randn_like(input_img, device=device).view(-1) * 0.1 + input_img.mean()
synth_img_flat.requires_grad_(True)

print(f"4D image shape: {synth_img_4d.shape}")
print(f"Flat image shape: {synth_img_flat.shape}")
print(f"Synthetic image device: {synth_img_4d.device}")
print(f"Synthetic image requires_grad: {synth_img_4d.requires_grad}")
print(f"Synthetic image range: [{synth_img_4d.min():.3f}, {synth_img_4d.max():.3f}]")
print(f"Target image range: [{input_img.min():.3f}, {input_img.max():.3f}]")

Using device: cuda
Note: Could not initialize distributed group: trying to initialize the default process group twice!
Will try alternative approach...
Target activations device: cuda:0
Target activations shape: torch.Size([1, 512, 1, 1])
Original image shape: torch.Size([1, 3, 224, 224])
4D image shape: torch.Size([1, 3, 224, 224])
Flat image shape: torch.Size([150528])
Synthetic image device: cuda:0
Synthetic image requires_grad: True
Synthetic image range: [-0.011, 0.872]
Target image range: [-2.118, 2.640]


In [51]:
# Set up optimizer for the synthetic image
# Try Muon with 4D image directly, then fallback to Adam if needed
try:
    # Try Muon with 4D image tensor directly
    param_groups = [
        dict(params=[synth_img_4d], use_muon=True, lr=0.02, weight_decay=0.01)
    ]
    optimizer = MuonWithAuxAdam(param_groups)
    optimizer_name = "Muon"
    synth_img = synth_img_4d  # Use 4D version
    use_4d_img = True
    print(f"Successfully set up Muon optimizer with 4D image tensor")
    print(f"Image shape: {synth_img.shape}")
except Exception as e:
    print(f"Muon optimizer failed with 4D image: {e}")
    print("Falling back to Adam optimizer...")
    # Fallback to Adam optimizer
    optimizer = torch.optim.Adam([synth_img_4d], lr=0.02, weight_decay=0.01)
    optimizer_name = "Adam"
    synth_img = synth_img_4d  # Use 4D version
    use_4d_img = True
    print(f"Successfully set up Adam optimizer")

print(f"Using {optimizer_name} optimizer for image optimization")
print(f"Using 4D image tensor: {use_4d_img}")

Successfully set up Muon optimizer with 4D image tensor
Image shape: torch.Size([1, 3, 224, 224])
Using Muon optimizer for image optimization
Using 4D image tensor: True


In [52]:
def metamer_loss(synth_img, target_activations):
    # Clear previous activations to avoid stale data
    activation.clear()
    
    # Forward pass through model to get activations (synth_img is always 4D)
    _ = model(synth_img)
    synth_activations = activation[layer_name]
    return F.mse_loss(synth_activations, target_activations)

In [53]:
n_steps = 300

# Verify everything is set up correctly before optimization
print(f"Before optimization:")
print(f"  synth_img.requires_grad: {synth_img.requires_grad}")
print(f"  synth_img.device: {synth_img.device}")
print(f"  target_activations.device: {target_activations.device}")

for step in range(n_steps):
    optimizer.zero_grad()
    loss = metamer_loss(synth_img, target_activations)
    loss.backward()
    optimizer.step()
    
    if step % 50 == 0:
        print(f"Step {step}, Loss: {loss.item():.6f}")
        
    # Check gradient flow on first step
    if step == 0:
        print(f"  synth_img.grad is not None: {synth_img.grad is not None}")
        if synth_img.grad is not None:
            print(f"  synth_img.grad.norm(): {synth_img.grad.norm().item():.6f}")

print(f"Optimization completed using {optimizer_name} optimizer!")

Before optimization:
  synth_img.requires_grad: True
  synth_img.device: cuda:0
  target_activations.device: cuda:0


RuntimeError: The size of tensor a (224) must match the size of tensor b (150528) at non-singleton dimension 3

In [ ]:
# Function to denormalize ImageNet normalized images for visualization
def denormalize_imagenet(tensor, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]):
    """Denormalize ImageNet normalized tensor for visualization"""
    mean = torch.tensor(mean).view(3, 1, 1)
    std = torch.tensor(std).view(3, 1, 1)
    return tensor * std + mean

# Convert 4D tensor to image for visualization (denormalize first)
result_img_norm = synth_img.detach().squeeze().cpu()
result_img = denormalize_imagenet(result_img_norm).permute(1, 2, 0).numpy()

plt.imshow(np.clip(result_img, 0, 1))
plt.title(f"Synthesized Metamer ({optimizer_name} Optimizer)")
plt.show()

In [ ]:
# Visualize the original and synthesized images side by side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Original image (denormalize ImageNet normalization)
orig_img_norm = input_img.detach().squeeze().cpu()
orig_img = denormalize_imagenet(orig_img_norm).permute(1, 2, 0).numpy()
ax1.imshow(np.clip(orig_img, 0, 1))
ax1.set_title(f"Original Image: {image_dict['filename_short']}")
ax1.axis('off')

# Synthesized metamer - convert 4D tensor to image (denormalize first)
result_img_norm = synth_img.detach().squeeze().cpu()
result_img = denormalize_imagenet(result_img_norm).permute(1, 2, 0).numpy()

ax2.imshow(np.clip(result_img, 0, 1))
ax2.set_title(f"Synthesized Metamer ({optimizer_name} Optimizer)")
ax2.axis('off')

plt.tight_layout()
plt.show()

# Print some statistics
print(f"Original image range (denormalized): [{orig_img.min():.3f}, {orig_img.max():.3f}]")
print(f"Synthesized image range (denormalized): [{result_img.min():.3f}, {result_img.max():.3f}]")
print(f"Original image range (normalized): [{orig_img_norm.min():.3f}, {orig_img_norm.max():.3f}]")
print(f"Synthesized image range (normalized): [{result_img_norm.min():.3f}, {result_img_norm.max():.3f}]")
print(f"Target activations shape: {target_activations.shape}")
print(f"Final loss: {loss.item():.6f}")
